Si se están clasificando datos y no se tiene un balance relativo en terminos de tamaño de las clases, el sesgo tenderá a irse más al lado de la clase más popular en el dataset y arrastrará al modelo, un ejemplo puede ser el caso en que un dataset de 100 muestras tenga 1 positiva y 99 negativas, facilmente el modelo dará una precisión de 99% por clasificar todo como negativo.

Existen diferentes formas de tratar con este problema desde basar diferentes metricas de refuerzo tales como el área bajo la curva AUC, hasta la regularización de modelos basados en árboles tales como randomforest o boosters como XGBoost


In [1]:
import pandas as pd

df = pd.read_excel("titanic3.xls")

df.head()
df = df.drop(columns=[
    "name",
    "ticket",
    "body",
    "boat",
    "home.dest",
    "cabin"
])

df = pd.get_dummies(df)

y = df.survived
X = df.drop(columns=["survived"])
X.head(3)

,pclass,age,sibsp,parch,fare,sex_female,sex_male,embarked_C,embarked_Q,embarked_S
0,1,29.0000,0,0,211.3375,True,False,False,False,True
1,1,0.9167,1,2,151.5500,False,True,False,False,True
2,1,2.0000,1,2,151.5500,True,False,False,False,True


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn import impute
num_cols = [
    "pclass",
    "age",
    "sibsp",
    "parch",
    "fare",
    "sex_female"
]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

imputer = impute.IterativeImputer()
imputed = imputer.fit_transform(X_train[num_cols])
X_train[num_cols] = imputed
imputed = imputer.transform(X_test[num_cols])
X_test[num_cols] = imputed

In [3]:
from sklearn.preprocessing import StandardScaler

cols = [
    "age",
    "pclass",
    "sibsp",
    "fare"
]

sca = StandardScaler()
X_train[cols] = sca.fit_transform(X_train[cols])
X_test[cols] = sca.transform(X_test[cols])

# Upsampling - remuestreo

Si se tiene el caso de un dataset desbalanceado en terminos de clases, en tareas de clasificación. Se puede implementar un remuestreo.

`Scikitlearn` provee la función resample en el paquete `sklearn.utils` , el cual recibe una estructura iterable como una lista, diccionario y `dataframe` de pandas y retorna un nuevo dataframe con las clases igualadas en número según la configuración del parametro n_samples

In [8]:
from sklearn.utils import resample

# Primero generamos una mascara filtrando todas las
# muestras marcadas como supervivients
mask = df.survived == 1

# Encontramos todos los pasajeros supervivientes y los fallecidos
surv_df = df[mask]
death_df = df[~mask]

# Balanceamos los elementos supervivientes de forma mas balanceadas basandose en el
# número de pasajeros fallecidos e indicamos que los reemplace de modo que retone un nuevo
# dataframe para concatenarlos con los registros de los fallecidos
# De esa formagenera un dataset mayor con un mejor balance
df_upsample = resample(
    surv_df,
    replace=True,
    n_samples=len(death_df),
    random_state=42
)

df2 = pd.concat([death_df, df_upsample])

df2.survived.value_counts()

survived
0    809
1    809
Name: count, dtype: int64

## Recurso adicional - imlearn

La librería `imbalanced-learn` también provee una clase RandomOverSampler que permite hacer un trabajo similar a resample de scikitlearn

# Downsampling

Una alternativa a remuestreo , el cual consiste en equiparar las clases faltantes basado el número de clases existentes , es el caso contrario


In [13]:
from sklearn.utils import resample

# De nuevo, filtramos las muestras que fueron marcadas como supervivientes
# y creamos dos nuevas dataframes identificando los supervivientes y los fallecidos
# y resampleamos para conseguir un nuevo dataset que sea concatenado con los supervivientes
mask = df.survived == 1
surv_df = df[mask]
death_df = df[~mask]


In [14]:
resample_df = resample(
    death_df,
    replace=False,
    n_samples=len(surv_df),
    random_state=42
)

df3 = pd.concat([surv_df, resample_df])
df3.survived.value_counts()

survived
1    500
0    500
Name: count, dtype: int64